In [ ]:
!pip install requests folium pandas geopandas -q 

## Análise Integrada dos Termos de Embargo do IBAMA e Candidaturas Eleitorais de 2022.
![](https://wikilai.fiquemsabendo.com.br/images/thumb/a/a8/Logo_principal_-_fiquem_sabendo.png/320px-Logo_principal_-_fiquem_sabendo.png)

In [6]:
#@title Importar GeoPandas
import geopandas as gpd
import pandas as pd
import requests
import numpy as np
import requests
import io
import folium
from folium.plugins import HeatMap


# Importação dos Dados Ibama e TSE

In [3]:
#@title GeoJson dos Embargos do Ibama maxfeatures = 100000
url_embargos_ibama ='https://siscom.ibama.gov.br/geoserver/publica/ows?service=WFS&version=1.0.0&request=GetFeature&typeName=publica:vw_brasil_adm_embargo_a&maxFeatures=100000&outputFormat=application%2Fjson'


In [7]:
#@title Ler GeoJson como GeoDataFrame
# Ler GeoDataFrame
# Adicionar um cabeçalho User-Agent para evitar o erro 403 Forbidden
# A URL do IBAMA pode bloquear requisições sem um User-Agent ou com User-Agents genéricos.

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36'
}

# Fazer a requisição HTTP usando requests
response = requests.get(url_embargos_ibama, headers=headers)
response.raise_for_status() # Lança uma exceção para erros HTTP (4xx ou 5xx)

# Usar io.BytesIO para tratar o conteúdo da resposta como um arquivo para o geopandas
gdf = gpd.read_file(io.BytesIO(response.content))

# Converter colunas para minúscula
gdf.columns = gdf.columns.str.lower()

In [5]:
gdf.shape

(89708, 29)

In [9]:
gdf['cpf_cnpj_infrator'].value_counts()

cpf_cnpj_infrator
054.408.922-73        191
009.150.172-53         25
04.680.054/0001-04     23
340.307.861-20         21
635.748.964-68         20
                     ... 
497.912.302-34          1
554.270.069-15          1
005.603.431-83          1
697.614.421-87          1
140.688.184-87          1
Name: count, Length: 66438, dtype: int64

## DADOS ELEIÇÃO

In [10]:
import requests, zipfile, io

# Exemplo: candidatos de 2022
url = "https://cdn.tse.jus.br/estatistica/sead/odsele/consulta_cand/consulta_cand_2022.zip"

# Baixar e abrir o zip
r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))

# Ver arquivos disponíveis
#print(z.namelist())

# Ler um CSV específico
df = pd.read_csv(z.open("consulta_cand_2022_BRASIL.csv"), sep=";", encoding="latin1")



In [14]:
gdf['data_tad'].nlargest(10)

19308   2026-06-19
3694    2026-05-06
41619   2026-05-06
43091   2026-05-06
61063   2026-05-06
66770   2026-05-06
5825    2026-05-05
8385    2026-05-05
77457   2026-05-05
29925   2026-05-04
Name: data_tad, dtype: datetime64[ms]

In [18]:
df['NR_CPF_CANDIDATO'].value_counts()

NR_CPF_CANDIDATO
-4              29
 2051040532      3
 1068199687      3
 61028347200     3
 87042967372     3
                ..
 12943037400     1
 16819657714     1
 29117682215     1
 66006040263     1
 20041373200     1
Name: count, Length: 28944, dtype: int64

In [19]:
df_candidatos=df

In [20]:
# Converter para string
df_candidatos["NR_CPF_CANDIDATO"] = df_candidatos["NR_CPF_CANDIDATO"].astype(str)

# Remover espaços e caracteres indesejados
df_candidatos["NR_CPF_CANDIDATO"] = df_candidatos["NR_CPF_CANDIDATO"].str.strip()

# Filtrar apenas CPFs com pelo menos 10 dígitos
df_candidatos = df[df["NR_CPF_CANDIDATO"].str.len() >= 10]

# Preencher com zeros à esquerda até 11 dígitos
df_candidatos["NR_CPF_CANDIDATO"] = df_candidatos["NR_CPF_CANDIDATO"].str.zfill(11)

df_candidatos['NR_CPF_CANDIDATO'].value_counts()


NR_CPF_CANDIDATO
02051040532    3
01068199687    3
61028347200    3
87042967372    3
26043947315    3
              ..
12943037400    1
16819657714    1
29117682215    1
66006040263    1
20041373200    1
Name: count, Length: 27617, dtype: int64

In [22]:
df["tamanho"] = df_candidatos["NR_CPF_CANDIDATO"].astype(str).str.len()
display(df[["NR_CPF_CANDIDATO","tamanho"]])

,NR_CPF_CANDIDATO,tamanho
0,-4,NaN
1,-4,NaN
2,16063112120,11.0
3,96637412872,11.0
4,83785302304,11.0
...,...,...
29309,16819657714,11.0
29310,29117682215,11.0
29311,66006040263,11.0
29312,20041373200,11.0


## REMOVER CNPJs

In [28]:
padrao_cnpj = r"\d{2}\.\d{3}\.\d{3}/\d{4}-\d{2}"
gdf = gdf.rename(columns={'cpf_cnpj_infrator':'NR_CPF_CANDIDATO'})
gdf_cpfs = gdf[~gdf["NR_CPF_CANDIDATO"].str.match(padrao_cnpj, na=False)]
gdf_cpfs.shape


(78242, 29)

In [29]:


gdf_cpfs["NR_CPF_CANDIDATO"] = gdf_cpfs["NR_CPF_CANDIDATO"].astype(str)


padrao_cpf = r"\d{3}\.\d{3}\.\d{3}-\d{2}"
gdf_cpfs = gdf[gdf["NR_CPF_CANDIDATO"].str.match(padrao_cpf, na=False)]
# Remove pontos e traços
gdf_cpfs["NR_CPF_CANDIDATO"] = gdf["NR_CPF_CANDIDATO"].str.replace(r"[.\-]", "", regex=True)

# Agora todos os CPFs ficam no formato 00000000000
gdf_cpfs["NR_CPF_CANDIDATO"]



0        41559444134
1        20326840397
2        00706981200
3        78135630263
5        03694545298
            ...     
89713    55427006915
89714    00560343183
89715    69761442187
89717    58877010991
89718    14068818487
Name: NR_CPF_CANDIDATO, Length: 69254, dtype: str

In [30]:
# Mantém apenas linhas com 11 dígitos
df_filtrado = df[df["tamanho"] >= 10]
df_filtrado.shape

(27939, 51)

In [31]:
df_filtrado[['NR_CPF_CANDIDATO', 'tamanho']]


,NR_CPF_CANDIDATO,tamanho
2,16063112120,11.0
3,96637412872,11.0
4,83785302304,11.0
5,60987222341,11.0
6,24531928334,11.0
...,...,...
29308,12943037400,11.0
29309,16819657714,11.0
29310,29117682215,11.0
29311,66006040263,11.0


## Infratores nas Eleições de 2022

In [53]:
# Configuração para mostrar todas as colunas
pd.set_option("display.max_columns", None)
Candidatos_Imfratores = df_candidatos[df_candidatos["NR_CPF_CANDIDATO"].isin(gdf_cpfs["NR_CPF_CANDIDATO"])]
print("Candidatos encontrados:")
display(Candidatos_Imfratores)

Candidatos encontrados:


,DT_GERACAO,HH_GERACAO,ANO_ELEICAO,CD_TIPO_ELEICAO,NM_TIPO_ELEICAO,NR_TURNO,CD_ELEICAO,DS_ELEICAO,DT_ELEICAO,TP_ABRANGENCIA,SG_UF,SG_UE,NM_UE,CD_CARGO,DS_CARGO,SQ_CANDIDATO,NR_CANDIDATO,NM_CANDIDATO,NM_URNA_CANDIDATO,NM_SOCIAL_CANDIDATO,NR_CPF_CANDIDATO,DS_EMAIL,CD_SITUACAO_CANDIDATURA,DS_SITUACAO_CANDIDATURA,TP_AGREMIACAO,NR_PARTIDO,SG_PARTIDO,NM_PARTIDO,NR_FEDERACAO,NM_FEDERACAO,SG_FEDERACAO,DS_COMPOSICAO_FEDERACAO,SQ_COLIGACAO,NM_COLIGACAO,DS_COMPOSICAO_COLIGACAO,SG_UF_NASCIMENTO,DT_NASCIMENTO,NR_TITULO_ELEITORAL_CANDIDATO,CD_GENERO,DS_GENERO,CD_GRAU_INSTRUCAO,DS_GRAU_INSTRUCAO,CD_ESTADO_CIVIL,DS_ESTADO_CIVIL,CD_COR_RACA,DS_COR_RACA,CD_OCUPACAO,DS_OCUPACAO,CD_SIT_TOT_TURNO,DS_SIT_TOT_TURNO
6,07/05/2026,03:30:13,2022,2,ELEIÇÃO ORDINÁRIA,1,546,Eleições Gerais Estaduais 2022,02/10/2022,ESTADUAL,MA,MA,MARANHÃO,7,DEPUTADO ESTADUAL,100001599551,90000,MARCOS ANTONIO DE CARVALHO CALDAS,MARCOS CALDAS,#NULO,24531928334,NÃO DIVULGÁVEL,12,APTO,PARTIDO ISOLADO,90,PROS,PARTIDO REPUBLICANO DA ORDEM SOCIAL,-1,#NULO,#NULO,#NULO,100001680999,PARTIDO ISOLADO,PROS,MA,09/02/1966,6818421104,2,MASCULINO,4,ENSINO FUNDAMENTAL COMPLETO,3,CASADO(A),3,PARDA,257,EMPRESÁRIO,4,NÃO ELEITO
382,07/05/2026,03:30:13,2022,2,ELEIÇÃO ORDINÁRIA,1,546,Eleições Gerais Estaduais 2022,02/10/2022,ESTADUAL,BA,BA,BAHIA,7,DEPUTADO ESTADUAL,50001609433,22123,JOÃO BATISTA ALVES PEREIRA,JOTA BATISTA,#NULO,92800505591,NÃO DIVULGÁVEL,12,APTO,PARTIDO ISOLADO,22,PL,PARTIDO LIBERAL,-1,#NULO,#NULO,#NULO,50001681438,PARTIDO ISOLADO,PL,BA,10/10/1977,78032040507,2,MASCULINO,8,SUPERIOR COMPLETO,3,CASADO(A),2,PRETA,131,ADVOGADO,5,SUPLENTE
2031,07/05/2026,03:30:13,2022,2,ELEIÇÃO ORDINÁRIA,1,546,Eleições Gerais Estaduais 2022,02/10/2022,ESTADUAL,PB,PB,PARAÍBA,7,DEPUTADO ESTADUAL,150001603132,44229,REGINALDO AMÉRICO TAVARES,REGINALDO AMÉRICO,#NULO,22622900406,NÃO DIVULGÁVEL,12,APTO,PARTIDO ISOLADO,44,UNIÃO,UNIÃO BRASIL,-1,#NULO,#NULO,#NULO,150001681176,PARTIDO ISOLADO,UNIÃO,PB,26/01/1961,668741252,2,MASCULINO,6,ENSINO MÉDIO COMPLETO,3,CASADO(A),1,BRANCA,257,EMPRESÁRIO,5,SUPLENTE
2037,07/05/2026,03:30:13,2022,2,ELEIÇÃO ORDINÁRIA,1,546,Eleições Gerais Estaduais 2022,02/10/2022,ESTADUAL,PB,PB,PARAÍBA,7,DEPUTADO ESTADUAL,150001603136,44777,JOSÉ ALEDSON DE SOUSA MOURA,DR. ALEDSON MOURA,#NULO,45843023320,NÃO DIVULGÁVEL,12,APTO,PARTIDO ISOLADO,44,UNIÃO,UNIÃO BRASIL,-1,#NULO,#NULO,#NULO,150001681176,PARTIDO ISOLADO,UNIÃO,CE,05/03/1974,40446940787,2,MASCULINO,8,SUPERIOR COMPLETO,3,CASADO(A),1,BRANCA,111,MÉDICO,5,SUPLENTE
2076,07/05/2026,03:30:13,2022,2,ELEIÇÃO ORDINÁRIA,1,546,Eleições Gerais Estaduais 2022,02/10/2022,ESTADUAL,PE,PE,PERNAMBUCO,7,DEPUTADO ESTADUAL,170001610745,77770,EVÂNGELA VIEIRA GALDINO VILELA DANTAS,EVÂNGELA VIEIRA,#NULO,93640935420,NÃO DIVULGÁVEL,12,APTO,PARTIDO ISOLADO,77,SOLIDARIEDADE,SOLIDARIEDADE,-1,#NULO,#NULO,#NULO,170001681528,PARTIDO ISOLADO,SOLIDARIEDADE,PE,30/10/1975,42552970841,4,FEMININO,8,SUPERIOR COMPLETO,3,CASADO(A),1,BRANCA,134,ASSISTENTE SOCIAL,5,SUPLENTE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25457,07/05/2026,03:30:13,2022,2,ELEIÇÃO ORDINÁRIA,1,546,Eleições Gerais Estaduais 2022,02/10/2022,ESTADUAL,PA,PA,PARÁ,6,DEPUTADO FEDERAL,140001601475,2044,MATEUS FERRAZ SOUZA,MATEUS FERRAZ,#NULO,41973585553,NÃO DIVULGÁVEL,12,APTO,PARTIDO ISOLADO,20,PSC,PARTIDO SOCIAL CRISTÃO,-1,#NULO,#NULO,#NULO,140001681115,PARTIDO ISOLADO,PSC,BA,01/08/1967,2574362852,2,MASCULINO,8,SUPERIOR COMPLETO,3,CASADO(A),1,BRANCA,257,EMPRESÁRIO,4,NÃO ELEITO
26884,07/05/2026,03:30:13,2022,2,ELEIÇÃO ORDINÁRIA,1,546,Eleições Gerais Estaduais 2022,02/10/2022,ESTADUAL,SP,SP,SÃO PAULO,6,DEPUTADO FEDERAL,250001613972,3365,MIGUEL GOMES FERNANDES,MIGUEL GOMES,#NULO,19701250206,NÃO DIVULGÁVEL,12,APTO,PARTIDO ISOLADO,33,PMN,PARTIDO DA MOBILIZAÇÃO NACIONAL,-1,#NULO,#NULO,#NULO,250001681693,PARTIDO ISOLADO,PMN,AC,15/04/1965,2128132461,2,MASCULINO

## Total de Infrações dos candidatos de 2022

In [52]:
Infracoes_Candidatos = gdf_cpfs[gdf_cpfs["NR_CPF_CANDIDATO"].isin(df_candidatos["NR_CPF_CANDIDATO"])]
print("Candidatos encontrados:")
display(Infracoes_Candidatos)

Candidatos encontrados:


,id,nom_pessoa,NR_CPF_CANDIDATO,seq_tad,numero_tad,data_tad,num_latitude_tad,num_longitude_tad,processo_tad,sig_uf,nom_municipio,num_auto_infracao,ser_auto_infracao,cod_municipio,cod_uf,status_tad,qtd_area_desmatada,data_cadastro_tad,des_infracao,serie_tad,sit_embarga_poligono,legislacao,artigo_legislacao,artigo,imagem_validacao,respeita_embargo,data_geom,orgao,geometry
281,vw_brasil_adm_embargo_a.282,JOSÉ LOPES JUNIOR,68395884249,A2J7UTZ8,A2J7UTZ8,2022-05-23,-9.42527800000000049,-67.1880560000000031,02001012769202218,AM,Lábrea,BUHPLB6M,NaN,1302405,13,AL,NaN,2022-05-23 12:34:42+00:00,Infração da Flora(Não Classificada-Móvel),NaN,None,None,None,None,None,None,2022-05-23,IBAMA,"MULTIPOLYGON (((-67.18805 -9.42528, -67.18805 ..."
1481,vw_brasil_adm_embargo_a.1482,JOSÉ LOPES JUNIOR,68395884249,FJOA5ZWZ,FJOA5ZWZ,2022-08-08,-9.16238899999999923,-67.0262129999999985,02001021565202260,AM,Lábrea,09YQZTF7,NaN,1302405,13,AL,69.75,2022-08-08 15:58:05+00:00,Infração da Flora(Não Classificada-Móvel),NaN,None,None,None,None,None,None,2022-08-08,IBAMA,"MULTIPOLYGON (((-67.03125 -9.16742, -67.03107 ..."
1513,vw_brasil_adm_embargo_a.1514,JOSÉ LOPES JUNIOR,68395884249,799661,799661,2018-03-30,-9.16846400000000017,-67.014008000000004,NaN,AM,Lábrea,9191380,E,1302405,13,AL,129.521,2018-03-30 12:49:32+00:00,Infração da Flora(Não Classificada-Móvel),E,None,None,None,None,None,None,2018-03-30,IBAMA,"MULTIPOLYGON (((-67.02049 -9.17034, -67.01609 ..."
1566,vw_brasil_adm_embargo_a.1567,MARCIO ANTONIO DE ARAUJO,66541077991,800117,800117,2018-01-12,-26.0091670000000015,-48.7916669999999968,NaN,SC,Garuva,9193244,E,4205803,42,AL,34.91,2018-01-12 14:15:46+00:00,Infração da Flora(Não Classificada-Móvel),E,None,None,None,None,None,None,2018-01-12,IBAMA,"MULTIPOLYGON (((-48.79262 -26.02088, -48.7925 ..."
2079,vw_brasil_adm_embargo_a.2080,CANDIDO HONORIO FERREIRA FILHO,04311396287,794426,794426,2019-02-05,-2.69083300000000003,-60.168332999999997,02005003504201802,AM,Manaus,NaN,NaN,1302603,13,AL,NaN,2019-02-05 13:54:00+00:00,NaN,E,None,None,None,None,None,None,2019-02-05,IBAMA,"MULTIPOLYGON (((-60.14249 -2.69092, -60.14258 ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87611,vw_brasil_adm_embargo_a.87612,ROSANI TEREZINHA PIRES DA COSTA DONADON,42021863204,440813,440813,2007-03-07,-12.6847499999999993,-60.1788889999999981,02502000278200736,RO,Vilhena,553776,D,1100304,11,AL,4.7,2007-03-07 14:05:00+00:00,Destruir ou danificar florestas ou demais form...,C,None,None,None,None,None,None,2007-03-07,IBAMA,"MULTIPOLYGON (((-60.18192 -12.68347, -60.18188..."
88610,vw_brasil_adm_embargo_a.88611,ROBSÃO DEMONTHI DE SOUZA MOREIRA,60266414249,829263,829263,2019-09-24,-9.32416699999999921,-60.8327779999999976,02055001084201915,MT,Colniza,9195214,E,5103254,51,AL,1182,2019-09-25 00:45:00+00:00,Infração da Flora(Não Classificada-Móvel),E,None,None,None,None,None,None,2019-09-24,IBAMA,"MULTIPOLYGON (((-60.83277 -9.32417, -60.83277 ..."
89046,vw_brasil_adm_embargo_a.89047,JOSÉ GERARDO OLIVEIRA DE ARRUDA FILHO,12197572334,569545,569545,2012-04-18,-12.473611,-48.677500000000002,02029000194201201,TO,Jaú do Tocantins,501278,D,1711506,17,AL,447.12,2012-04-18 19:20:00+00:00,Explorar ou danificar floresta ou qualquer tip...,C,None,None,None,None,None,None,2012-04-18,IBAMA,"MULTIPOLYGON (((-48.64351 -12.49174, -48.65206..."
89319,vw_brasil_adm_embargo_a.89320,SEBASTIÃO VIEIRA DA SILVA,28905504949,25964,25964,2017-04-05,-27.5991670000000013,-48.5936110000000028,NaN,SC,Florianópolis,9057512,E,4205407,42,AL,NaN,2017-04-05 18:24:15+00:00,Infração da Fauna(Não Classificada-Móvel),E,None,None,None,None,None,None,2017-04-05,IBAMA,"MULTIPOLYGON (((-48.5936 -27.59917, -48.5936 -..."


In [36]:
# Configuração para mostrar todas as colunas
#Exibir apenas as linhas correspondentes:
pd.set_option("display.max_columns", None)
df_candidatos[df_candidatos['NR_CPF_CANDIDATO'] == '68395884249']


,DT_GERACAO,HH_GERACAO,ANO_ELEICAO,CD_TIPO_ELEICAO,NM_TIPO_ELEICAO,NR_TURNO,CD_ELEICAO,DS_ELEICAO,DT_ELEICAO,TP_ABRANGENCIA,...,CD_GRAU_INSTRUCAO,DS_GRAU_INSTRUCAO,CD_ESTADO_CIVIL,DS_ESTADO_CIVIL,CD_COR_RACA,DS_COR_RACA,CD_OCUPACAO,DS_OCUPACAO,CD_SIT_TOT_TURNO,DS_SIT_TOT_TURNO
11244,07/05/2026,03:30:13,2022,2,ELEIÇÃO ORDINÁRIA,1,546,Eleições Gerais Estaduais 2022,02/10/2022,ESTADUAL,...,8,SUPERIOR COMPLETO,1,SOLTEIRO(A),1,BRANCA,131,ADVOGADO,5,SUPLENTE
